# Stage 1: Select Position Openers

Select wallets whose opening BUYs are worth copying.
Uses volatility-based wallet metrics + threshold scoring (matching reference notebook).

Grid-search over selection thresholds to maximize **copyable PnL from opening buys** on the validation split.

**Output:** `stage1_result.json` with best selection params.

In [89]:
%load_ext autoreload
%autoreload 2

import time

import numpy as np
import pandas as pd

from lib import (
    load_trades,
    split_data,
    compute_copyable_notional,
    compute_opening_metrics,
    evaluate_wallet_group,
    evaluate_wallet_group_openers,
    select_copyable_group,
    run_grid_search,
    save_stage_result,
    DEFAULT_TAGS,
)
from polymarket_analysis.wallet_selection.volatility import compute_wallet_metrics

pd.options.display.float_format = "{:.4f}".format
pd.options.display.max_rows = 100

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load data

In [90]:
df_full = load_trades()
df_full = compute_copyable_notional(df_full)
df_train, df_val, df_test = split_data(df_full, method='chronological')

Markets: 1877548
Filtered markets for {'Weather'}: 87967
Loading 16 trade shards...
Total trades loaded: 13,603,198
Unique wallets: 4,054
Date range: 2025-01-09 15:32:39+00:00 -> 2026-07-22 05:47:10+00:00
Chronological split: train <= 2026-05-19T00:00:00Z, val <= 2026-06-20T00:00:00Z, test > 2026-06-20T00:00:00Z
Method: chronological  |  Unique end dates: 107  (train=42, val=32, test=33)

  Train:  4,136,747 trades  (14,688 markets)
  Val:    4,972,752 trades  (19,692 markets)
  Test:   4,493,699 trades  (19,352 markets)
  Total: 13,603,198 trades  (53,732 markets)


In [91]:
df_test['end_date_iso'].min()

'2026-06-21T00:00:00Z'

## Compute wallet metrics on training data

In [92]:
wallet_vol, _ = compute_wallet_metrics(df_train)

wallet_vol["copyable_pnl_factor"] = np.clip(
    wallet_vol["copyable_pnl"] / wallet_vol["total_pnl"].replace(0, np.nan),
    0, 1.0,
).fillna(0.0)
wallet_vol["copyable_roi"] = wallet_vol["average_roi"] * wallet_vol["copyable_pnl_factor"]

opening_metrics = compute_opening_metrics(df_train)
wallet_vol = wallet_vol.merge(opening_metrics, on="wallet", how="left")
for c in ["opening_roi", "opening_pnl", "opening_copyable_roi", "opening_copyable_pnl"]:
    wallet_vol[c] = wallet_vol[c].fillna(0.0)

print(f"Wallets with metrics: {len(wallet_vol)}")
wallet_vol[["wallet", "buy_roi", "opening_roi", "opening_pnl", "opening_copyable_pnl", "copyable_pnl", "num_buckets"]].head(10)

Wallets with metrics: 3159


,wallet,buy_roi,opening_roi,opening_pnl,opening_copyable_pnl,copyable_pnl,num_buckets
0,0x0054ee7dfb882d2d016fa13ef5f5cdb3b0ebcf1f,0.0159,0.0180,8.2562,-0.1541,-0.0365,33
1,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0.0619,0.0589,43.0043,-7.3919,0.6293,195
2,0x00833cc2d777e6f2fc8437679124024ae6468cb1,NaN,-1.0000,-1.0000,0.0000,0.0000,2
3,0x0141be702d272f17666e280303ad44e7bc0cc2da,0.6776,0.6033,59.3074,55.1701,115.4494,34
4,0x015be8bad14c79d2722a0bd8bbe0cd93b905556d,NaN,-0.0877,-17.1649,-21.0658,-20.9264,299
5,0x01a5fb1fa13f378138a31382c8364b5d4e2b0e36,0.0090,0.0194,5.7705,1.4397,8.6100,23
6,0x01a68281185e728ba0fef6245008bf8af68a59b0,0.0094,-0.0124,-21.0843,8.9906,4.4309,598
7,0x01ced860d8dca5d7987579d2a2635df8520d27a2,0.2043,0.1279,494.0007,46.2288,472.9760,911
8,0x01d94480e2a96cdd01fed071878b1adf82e0acd0,NaN,-0.0042,-42.3820,-26.9492,62.4920,1173
9,0x01f2a8baabe17c2541d1e3091220991f257ac3de,0.0099,0.0036,182.4516,-772.7195,-3425.7993,39804


## Baseline selection (reference defaults)

In [93]:
copyable_group = select_copyable_group(
    wallet_vol,
    min_buy_roi=0.05,
    min_num_buckets=20,
    min_num_markets=15,
    max_drawdown_to_pnl=0.20,
    max_top_market_pnl_pct=0.25,
    max_market_pnl_hhi=0.30,
    min_total_notional=5_000,
    min_opening_roi=0.0,
    min_opening_pnl=0,
    min_opening_copyable_roi=0.0,
)
print(f"Copyable group: {len(copyable_group)} wallets")
show_cols = ["wallet", "opening_roi", "opening_copyable_roi", "opening_pnl",
             "opening_copyable_pnl", "copyable_pnl", "buy_roi", "num_buckets"]
copyable_group[[c for c in show_cols if c in copyable_group.columns]].head(15)

Copyable group: 33 wallets


,wallet,opening_roi,opening_copyable_roi,opening_pnl,opening_copyable_pnl,copyable_pnl,buy_roi,num_buckets
0,0x07d601375c9bbb9037ad3c7a8f8fa0deff8164fb,0.2447,0.3557,424.0297,161.2472,142.4246,0.1375,296
1,0x7231a52f9de4fda5218d0e63f30a3499a4535afe,0.1940,0.3261,376.7979,206.5682,1335.9166,0.3103,602
2,0x35aff83368c69c47af04ee2d99330154f22f1ca6,0.5797,0.3257,493.6992,140.3441,206.5448,0.6693,489
3,0xaafc62dace9a5fef1b4b8028ca37adeb9edbec67,0.3390,0.2550,997.9247,309.3140,118.4968,0.3719,494
4,0x46532d38063a22045404aeead6a9f3c49e75fc2b,0.1316,0.2021,386.5703,204.4402,337.4733,0.0804,115
5,0x0e09d1f32963451855e429c384be6499dd0e5eef,0.1982,0.1933,404.3683,229.4965,346.3436,0.1949,98
6,0x919698b19427cbe6945b0dc823f2d9e126a4d934,0.1032,0.1542,457.0521,167.2978,455.0186,0.1450,688
7,0x45e606f7849330adad37875f835fbdd1b7868fca,0.0774,0.1320,588.9154,288.1546,723.0666,0.1512,941
8,0x8fb431f5057112cfdb0e7f566b4b687cb69cb9ed,0.1057,0.1259,1418.8071,459.0128,4198.6197,0.2913,1407
9,0xe5b1377410ddd77dc3bf6fd5172e13fdc31330b2,0.1054,0.1225,565.4660,130.3783,222.7708,0.1465,492


## Baseline evaluation (reference format)

In [94]:
wallet_set = set(copyable_group["wallet"])

for split_name, df_split in [("TRAIN", df_train), ("VAL", df_val), ("TEST", df_test)]:
    evaluate_wallet_group(df_split, wallet_set, label=f"{split_name} copyable group")


*** TRAIN copyable group ***
  Open  : wallet_pnl=  21313.33  roi=0.0889  |  copyable_pnl=   4655.83  roi=0.0818
  Total : wallet_pnl=  51600.59  roi=0.1144  |  copyable_pnl=  15647.78  roi=0.1147

*** VAL copyable group ***
  Open  : wallet_pnl=  10675.33  roi=0.0323  |  copyable_pnl=   1434.94  roi=0.0173
  Total : wallet_pnl=  24981.64  roi=0.0378  |  copyable_pnl=   3841.11  roi=0.0197

*** TEST copyable group ***
  Open  : wallet_pnl=  11400.80  roi=0.0453  |  copyable_pnl=     70.45  roi=0.0011
  Total : wallet_pnl=  18876.28  roi=0.0383  |  copyable_pnl=    435.86  roi=0.0030


## Grid search

Vary selection thresholds to maximize copyable PnL from opening buys on the validation split.

In [95]:
param_grid = dict(
    min_buy_roi=[0.05, 0.07, 0.1],
    min_num_buckets=[15],
    min_num_markets=[10, 20],
    max_drawdown_to_pnl=[0.1, 0.2, 0.3],
    max_top_market_pnl_pct=[1],
    max_market_pnl_hhi=[0.30],
    min_total_notional=[1_000],
    min_opening_roi=[0.05, 0.07, 0.1, 0.2],
    min_opening_pnl=[200],
    min_opening_copyable_roi=[0.05, 0.07, 0.1],
)

print(f"Grid: {np.prod([len(v) for v in param_grid.values()]):.0f} combos")

Grid: 216 combos


In [96]:
res_df = run_grid_search(param_grid, wallet_vol, df_val)
# print(f'open_copyable_pnl: {res_df["open_copyable_pnl"].max():.0f}, open_copyable_roi: {res_df["open_copyable_pnl"].max():.4f}')
res_df.head()

best_row = res_df.iloc[0]
best_params = {k: best_row[k] for k in param_grid.keys()}
best_group = select_copyable_group(wallet_vol, **best_params)

print(f"Best config (val open copyable_pnl={best_row['open_copyable_pnl']:.2f}):")
print(best_params)
print(f"  wallets: {best_row['wallets']:.0f}  open_wallets: {best_row['open_wallets']:.0f}")

Grid: 216 combos, 8 workers
  [100/216] 12.4s elapsed
  [200/216] 24.3s elapsed
  [216/216] 25.6s elapsed
Done: 216 configs in 25.6s
Best config (val open copyable_pnl=3230.97):
{'min_buy_roi': np.float64(0.07), 'min_num_buckets': np.float64(15.0), 'min_num_markets': np.float64(10.0), 'max_drawdown_to_pnl': np.float64(0.3), 'max_top_market_pnl_pct': np.float64(1.0), 'max_market_pnl_hhi': np.float64(0.3), 'min_total_notional': np.float64(1000.0), 'min_opening_roi': np.float64(0.05), 'min_opening_pnl': np.float64(200.0), 'min_opening_copyable_roi': np.float64(0.05)}
  wallets: 58  open_wallets: 46


In [97]:
print('top 10 results')
res_df.head(10)

top 10 results


,min_buy_roi,min_num_buckets,min_num_markets,max_drawdown_to_pnl,max_top_market_pnl_pct,max_market_pnl_hhi,min_total_notional,min_opening_roi,min_opening_pnl,min_opening_copyable_roi,open_copyable_pnl,open_wallet_pnl,open_wallets,total_copyable_pnl,wallets,elapsed
96,0.0700,15,10,0.3000,1,0.3000,1000,0.0500,200,0.0500,3230.9652,30887.7213,46,9468.7141,58,1.1398
132,0.0700,15,20,0.3000,1,0.3000,1000,0.0500,200,0.0500,3197.9456,30397.2609,44,9380.8048,55,1.3076
28,0.0500,15,10,0.3000,1,0.3000,1000,0.0700,200,0.0500,2987.9325,30026.0928,46,8708.6151,58,0.9490
99,0.0700,15,10,0.3000,1,0.3000,1000,0.0700,200,0.0500,2982.1130,29786.6630,45,8899.0631,57,1.0191
63,0.0500,15,20,0.3000,1,0.3000,1000,0.0700,200,0.0500,2954.9129,29535.6324,44,8620.7058,55,1.2247
135,0.0700,15,20,0.3000,1,0.3000,1000,0.0700,200,0.0500,2949.0934,29296.2027,43,8811.1538,54,1.1865
29,0.0500,15,10,0.3000,1,0.3000,1000,0.1000,200,0.0500,2842.1872,27559.7079,42,7102.4786,52,1.1464
177,0.1000,15,10,0.3000,1,0.3000,1000,0.1000,200,0.0500,2841.0221,22786.1329,36,8151.5362,46,0.9865
102,0.0700,15,10,0.3000,1,0.3000,1000,0.1000,200,0.0500,2836.3677,27320.2781,41,7292.9267,51,1.0091
208,0.1000,15,20,0.3000,1,0.3000,1000,0.1000,200,0.0500,2831.3233,22747.3156,35,8122.0802,44,0.8851


## Stage 1 results

In [98]:
if best_group is not None and not best_group.empty:
    print(f"Copyable group: {len(best_group)} wallets")
    cols = [c for c in ["wallet", "opening_roi", "opening_copyable_roi", "opening_pnl",
                         "opening_copyable_pnl", "copyable_pnl", "buy_roi", "num_buckets"]
            if c in best_group.columns]
    print(best_group[cols].head(15).to_string())

Copyable group: 58 wallets
                                        wallet  opening_roi  opening_copyable_roi  opening_pnl  opening_copyable_pnl  copyable_pnl  buy_roi  num_buckets
0   0x7384822f7f2476e928168a3f360728d1ec041ebd       1.1037                1.5417     506.8617              245.2449      167.9121   0.6410          160
1   0x74957ea27ac4fbdee46d861fdae357859ff67fcf       0.5935                1.5025     961.3764              228.4146      271.2115   0.8160         6655
2   0xd8798d9aa2c7c05bfefaf6436e8cbd4cd5de7bc3       0.9124                0.9199     408.4962              144.7854       13.9663   0.5952          252
3   0x76305b2a31e7ec35189650c93e8df2a15b92789d       0.7246                0.7884     490.1535              258.8270      447.4757   1.4334          351
4   0xa493ef05d1e4722c0689547cef88f6e2ff821b55       0.4406                0.7229     323.4668              119.7493      234.6346   0.5965          855
5   0xa776cf00b1529e295ce8c9aa085640d9d97262a0       0.

In [99]:
wallet_set = set(best_group["wallet"])
print(f"\nSelected {len(wallet_set)} wallets")
for split_name, df_split in [("TRAIN", df_train), ("VAL", df_val), ("TEST", df_test)]:
    evaluate_wallet_group(df_split, wallet_set, label=f"{split_name} copyable group")


Selected 58 wallets

*** TRAIN copyable group ***
  Open  : wallet_pnl=  33604.93  roi=0.1723  |  copyable_pnl=   9503.06  roi=0.1715
  Total : wallet_pnl=  99355.75  roi=0.1848  |  copyable_pnl=  29167.43  roi=0.1750

*** VAL copyable group ***
  Open  : wallet_pnl=  30887.72  roi=0.1056  |  copyable_pnl=   3230.97  roi=0.0440
  Total : wallet_pnl=  60018.46  roi=0.0752  |  copyable_pnl=  12304.44  roi=0.0584

*** TEST copyable group ***
  Open  : wallet_pnl=  21565.57  roi=0.0945  |  copyable_pnl=   2256.55  roi=0.0399
  Total : wallet_pnl=  37824.47  roi=0.0580  |  copyable_pnl=   2940.16  roi=0.0203


## Save stage 1 result

In [100]:
import json
from datetime import datetime, timezone
from pathlib import Path

wallet_cols = [
    "wallet", "buy_roi", "opening_roi", "opening_pnl",
    "opening_copyable_roi", "opening_copyable_pnl",
    "copyable_pnl", "copyable_roi",
    "num_buckets", "num_markets", "total_notional", "total_pnl",
    "max_drawdown_to_pnl", "top_market_pnl_pct", "market_pnl_hhi",
    "wallet_quality",
]
wallet_records = best_group[[c for c in wallet_cols if c in best_group.columns]].to_dict(orient="records")


def _convert(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


wallet_records = [{k: _convert(v) for k, v in w.items()} for w in wallet_records]

metadata = {
    "type": "openers",
    "tags": sorted(DEFAULT_TAGS),
    "run_timestamp": datetime.now(timezone.utc).isoformat(),
    "n_wallets_selected": len(best_group),
    "n_wallets_total": len(wallet_vol),
}

payload = {
    "stage": 1,
    "best_params": {k: _convert(v) for k, v in best_params.items()},
    "best_open_copyable_pnl": float(best_row["open_copyable_pnl"]),
    "metadata": metadata,
    "wallets": wallet_records,
}

out_path = Path("stage1_result.json")
with open(out_path, "w") as f:
    json.dump(payload, f, indent=2)
print(f"Saved stage 1 implied result -> {out_path.resolve()}")

Saved stage 1 implied result -> /Users/vobornij/projects/polymarket/notebooks/wallet_selection/stage1_result.json


In [101]:
df = df_test[
    (df_test["wallet"].isin(wallet_set))
    & (df_test["side"] == "BUY") & (df_test["position"] == df_test["quantity"])
    ]
len(df)

22249

In [102]:
df['copyable_pnl'].sum()

np.float64(2256.5532887873605)